# 🚀 TASK 6.4 & TASK 6.5: MA TRẬN ABLATION STUDY VÀ HIỆU CHUẨN GATE 4 ĐIỀU KIỆN
### Đánh giá đối chứng 5 cấu hình M1–M5, hiệu chuẩn nhiệt độ và tuyển chọn mô hình đại diện

> **Mã nhiệm vụ**: `Task 6.4` & `Task 6.5` *(Milestone 6 - Tuần 6)*  
> **Người chủ trì**: **Thành viên C (Training & Evaluation Lead)**  
> **Người phối hợp**: **Thành viên B (Architecture Lead)**, **Thành viên A (Data & Infra Lead)**  
> **Môi trường thực thi**: Kaggle Server GPU (NVIDIA T4 / P100 / RTX PRO 6000 Blackwell)  
> **Thời gian chạy dự kiến**: ~3 đến 5 phút (Inference-only, không huấn luyện lại)  

---
### 📌 NGUYÊN TẮC KHOA HỌC:
1. **Bảo vệ tuyệt đối 18 tập test Task 5.1**: Chỉ sử dụng tập validation đa miền độc lập (540 mẫu). 18 tập test benchmark hoàn toàn cách ly, không rò rỉ dữ liệu.
2. **Phân tách hai phân vùng độc lập**: 50% `val_calibration` (tối ưu nhiệt độ $T$ và quét ngưỡng $T^*$) và 50% `val_selection` (chấm điểm xếp hạng khách quan).
3. **Kiểm định Cổng Gate 4 điều kiện**: Đối chiếu tương đối với mốc sàn baseline gốc đã xác lập tại Task 6.2.

In [ ]:
# ==============================================================================
# 1. THIẾT LẬP MÃ NGUỒN VÀ MÔI TRƯỜNG THỰC THI TRÊN KAGGLE
# ==============================================================================
import os
import sys
import glob
import shutil
import zipfile

print('=' * 80)
print('BƯỚC 1: ĐỒNG BỘ MÃ NGUỒN FATFORMER-XLA TRÊN KAGGLE')
print('=' * 80)

WORK_DIR = '/kaggle/working/fatformer-xla'
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

# Tim kiem ma nguon tu Kaggle Input
zip_cands = glob.glob('/kaggle/input/**/fatformer_code.zip', recursive=True) + \
            glob.glob('/kaggle/input/**/fatformer-code*.zip', recursive=True)
train_cands = glob.glob('/kaggle/input/**/train_crossdomain.py', recursive=True)

if zip_cands:
    print(f'[*] Giai nen ma nguon tu goi: {zip_cands[0]}')
    with zipfile.ZipFile(zip_cands[0], 'r') as zf:
        zf.extractall(WORK_DIR)
elif train_cands:
    root_cand = os.path.abspath(os.path.join(os.path.dirname(train_cands[0]), '..', '..'))
    print(f'[*] Sao chep ma nguon tu thu muc: {root_cand}')
    shutil.copytree(root_cand, WORK_DIR, dirs_exist_ok=True)
else:
    print('[*] Tu dong clone ma nguon phien ban moi cap nhat tu GitHub...')
    !git clone -b hoang https://github.com/LeeVHoangtk3/fatformer-xla.git {WORK_DIR}

# Xu ly tokenizer BPE offline
clip_dir = os.path.join(WORK_DIR, 'src', 'models', 'clip')
os.makedirs(clip_dir, exist_ok=True)
target_bpe = os.path.join(clip_dir, 'bpe_simple_vocab_16e6.txt.gz')
bpe_cands = glob.glob('/kaggle/**/bpe_simple_vocab_16e6.txt.gz', recursive=True)
if not os.path.exists(target_bpe) and bpe_cands:
    shutil.copy2(bpe_cands[0], target_bpe)
    print(f'[✓] Da dong bo BPE vocab offline: {target_bpe}')

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print('[✓] Hoan tat thiet lap moi truong ma nguon!')


In [ ]:
# ==============================================================================
# 2. ĐỊNH VỊ CÁC TÀI NGUYÊN CHECKPOINT M1–M5 VÀ KHO DỮ LIỆU ĐỐI CHỨNG
# ==============================================================================
def find_file(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return os.path.abspath(matches[0])
    return None

print('=' * 80)
print('BƯỚC 2: ĐỊNH VỊ CHECKPOINTS VÀ DỮ LIỆU ĐA MIỀN')
print('=' * 80)

clip_path = find_file([
    '/kaggle/input/**/ViT-L-14.pt',
    f'{WORK_DIR}/DATASET/pretrained/ViT-L-14.pt',
    f'{WORK_DIR}/pretrained/ViT-L-14.pt',
    '/kaggle/working/**/ViT-L-14.pt'
])
print(f'  - CLIP Backbone: {clip_path}')

m1_ckpt = find_file([
    '/kaggle/input/**/fatformer_4class_ckpt.pth',
    f'{WORK_DIR}/DATASET/pretrained/fatformer_4class_ckpt.pth',
    f'{WORK_DIR}/fatformer_4class_ckpt.pth'
])
print(f'  - M1 (Baseline 4-class): {m1_ckpt}')

m2_ckpt = find_file([
    '/kaggle/input/**/fatformer_plan_a_best_weights.pth',
    f'{WORK_DIR}/DATASET/checkpoints/fatformer_plan_a_best_weights.pth',
    f'{WORK_DIR}/DATASET/checkpoints/fatformer_plan_a_best.pth'
])
print(f'  - M2 (Plan A Finetune): {m2_ckpt}')

m3_ckpt = find_file([
    '/kaggle/input/**/m3/crossdomain_model_best.pth',
    '/kaggle/input/**/crossdomain_model_best.pth',
    f'{WORK_DIR}/DATASET/checkpoints/m3/crossdomain_model_best.pth',
    '/kaggle/working/checkpoints/m3/crossdomain_model_best.pth'
])
print(f'  - M3 (Data Only): {m3_ckpt}')

m4_ckpt = find_file([
    '/kaggle/input/**/m4/crossdomain_model_best.pth',
    f'{WORK_DIR}/DATASET/checkpoints/m4/crossdomain_model_best.pth',
    '/kaggle/working/checkpoints/m4/crossdomain_model_best.pth'
])
print(f'  - M4 (Distill + Cons): {m4_ckpt}')

m5_ckpt = find_file([
    '/kaggle/input/**/m5/crossdomain_model_best.pth',
    f'{WORK_DIR}/DATASET/checkpoints/m5/crossdomain_model_best.pth',
    '/kaggle/working/checkpoints/m5/crossdomain_model_best.pth'
])
print(f'  - M5 (LoRA ViT Adapter): {m5_ckpt}')

manifest_path = find_file([
    f'{WORK_DIR}/DATASET/manifests/data_manifest.json',
    '/kaggle/input/**/data_manifest.json'
])
print(f'  - Manifest du lieu: {manifest_path}')

# Kiem tra duong dan anh trong manifest co ton tai tren dia khong
import json
if manifest_path:
    with open(manifest_path, 'r', encoding='utf-8') as f:
        m_data = json.load(f)
    val_recs = [r for r in m_data.get('records', []) if r.get('split') == 'val']
    existing_val = sum(1 for r in val_recs if os.path.exists(r.get('file_path', '')))
    print(f'[✓] Kiem tra tap validation: {existing_val}/{len(val_recs)} anh ton tai san tren dia Kaggle!')
    if existing_val == 0:
        # Tu dong remap duong dan neu folder dataset nam o vi tri khac
        real_staging_dirs = glob.glob('/kaggle/input/**/diffusion_staging', recursive=True)
        if real_staging_dirs:
            base_s = real_staging_dirs[0]
            print(f'[*] Phat hien thu muc anh thuc te: {base_s}. Dang tu dong cap nhat duong dan...')
            for r in m_data.get('records', []):
                orig_p = r.get('file_path', '')
                if 'diffusion_staging' in orig_p:
                    sub_p = orig_p.split('diffusion_staging')[-1].lstrip('/\\')
                    new_p = os.path.join(base_s, sub_p)
                    if os.path.exists(new_p):
                        r['file_path'] = new_p
            with open(manifest_path, 'w', encoding='utf-8') as f:
                json.dump(m_data, f, indent=2)
            new_val_exist = sum(1 for r in m_data.get('records', []) if r.get('split') == 'val' and os.path.exists(r.get('file_path', '')))
            print(f'[✓] Cap nhat duong dan thanh cong: {new_val_exist}/{len(val_recs)} anh san sang!')


In [ ]:
# ==============================================================================
# 3. THỰC THI ĐO ĐẠC MA TRẬN ABLATION (TASK 6.5) & HIỆU CHUẨN GATE (TASK 6.4)
# ==============================================================================
import subprocess

OUTPUT_LOGS = '/kaggle/working/output_logs'
os.makedirs(OUTPUT_LOGS, exist_ok=True)

cmd = [
    'python', '-u', f'{WORK_DIR}/tools/eval_crossdomain_ablation.py',
    '--manifest', manifest_path or f'{WORK_DIR}/DATASET/manifests/data_manifest.json',
    '--baseline_metrics', f'{WORK_DIR}/DATASET/logs/val_crossdomain_baseline_metrics.json',
    '--output_dir', OUTPUT_LOGS,
    '--batch_size', '16',
    '--device', 'cuda',
    '--seed', '42',
]

if clip_path:
    cmd.extend(['--clip_path', clip_path])
if m1_ckpt:
    cmd.extend(['--m1_ckpt', m1_ckpt])
if m2_ckpt:
    cmd.extend(['--m2_ckpt', m2_ckpt])
if m3_ckpt:
    cmd.extend(['--m3_ckpt', m3_ckpt])
if m4_ckpt:
    cmd.extend(['--m4_ckpt', m4_ckpt])
if m5_ckpt:
    cmd.extend(['--m5_ckpt', m5_ckpt])

print('=' * 80)
print('BƯỚC 3: TIẾN HÀNH SUY LUẬN MA TRẬN ABLATION VÀ KIỂM ĐỊNH CỔNG GATE')
print(f'[*] Lenh thuc thi: {" ".join(cmd)}')
print('=' * 80)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Tien trinh ket thuc voi ma loi: {process.returncode}')
else:
    print('\n[✓] TIẾN TRÌNH ĐO ĐẠC VÀ HIỆU CHUẨN ĐÃ HOÀN TẤT THÀNH CÔNG!')


In [ ]:
# ==============================================================================
# 4. HIỂN THỊ TRỰC QUAN KẾT QUẢ VÀ ĐÓNG GÓI BÁO CÁO TẢI VỀ
# ==============================================================================
import pandas as pd
from IPython.display import display, Markdown

ablation_csv = os.path.join(OUTPUT_LOGS, 'crossdomain_ablation_matrix.csv')
report_json = os.path.join(OUTPUT_LOGS, 'calibration_and_selection_report.json')

if os.path.exists(ablation_csv):
    df_abl = pd.read_csv(ablation_csv)
    display(Markdown('### 📊 BẢNG TỔNG HỢP MA TRẬN ABLATION STUDY (TASK 6.5):'))
    display(df_abl)

if os.path.exists(report_json):
    with open(report_json, 'r', encoding='utf-8') as f:
        r_data = json.load(f)
    display(Markdown(f"### 🏆 KẾT LUẬN TUYỂN CHỌN THEO GATE 4 ĐIỀU KIỆN (TASK 6.4):"))
    display(Markdown(f"- **Mô hình đại diện được chọn**: `{r_data['metadata']['selected_model']}`"))
    display(Markdown(f"- **Điểm Composite Score đạt được**: `{r_data['metadata']['selected_composite_score']:.2f}%`"))

# Dong goi ket qua nhe va file trong so rut gon ra /kaggle/working/
export_zip = '/kaggle/working/task_6.4_6.5_results.zip'
with zipfile.ZipFile(export_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_LOGS):
        for fn in files:
            fp = os.path.join(root, fn)
            arcname = os.path.relpath(fp, OUTPUT_LOGS)
            zf.write(fp, arcname)

zip_size_mb = os.path.getsize(export_zip) / (1024 * 1024)
print(f'[✓] Da dong goi thanh cong toan bo ket qua ra: {export_zip} ({zip_size_mb:.2f} MB)')
print('Ban co the tai truc tiep tep task_6.4_6.5_results.zip ve may de nghiem thu!')
